In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/cat-in-the-dat/sample_submission.csv
/kaggle/input/competitions/cat-in-the-dat/train.csv
/kaggle/input/competitions/cat-in-the-dat/test.csv


이번 장에서는 베이스라인 모델 자체의 성능을 높일 것.

그전에는 다른 모델을 활용하는 방식으로 성능 향상을 기대했므로 차이가 있다.

> 중요합니다~!

성능 향상을 위해 다음 주안점에 집중

1. 피쳐 맞춤 인코딩 : 이진/순서형 피처들에게는 수작업 인코딩, 순서형 피처는 ordinal, 명목형, 날짜에는 원핫
2. 피쳐 스케일링 : 순서형 피쳐에만 적용 - 피처 간의 값의 범위를 획일화해야하는 피쳐에만
3. 하이퍼파라미터 최적화 : 그리드 서치를 활용

In [2]:
import pandas as pd

data_path = '/kaggle/input/competitions/cat-in-the-dat/'

train = pd.read_csv(data_path + 'train.csv', index_col='id')
test = pd.read_csv(data_path + 'test.csv', index_col='id')

submission = pd.read_csv(data_path+ 'sample_submission.csv', index_col='id')

피쳐 엔지니어링 시작:

Train/test concat -> 이진 피처 수작업 -> 순서형 피처 수작업 -> 명목형, 날짜 피처 분리 -> 해당 피처들 원핫 인코딩 -> 다시 모두 합치기 (피쳐 스케일링까지 끝내고)

In [3]:
all_data = pd.concat([train, test])
all_data = all_data.drop('target', axis=1)

In [4]:
all_data.head()

,bin_0,bin_1,bin_2,bin_3,bin_4,nom_0,nom_1,nom_2,nom_3,nom_4,...,nom_8,nom_9,ord_0,ord_1,ord_2,ord_3,ord_4,ord_5,day,month
id,,,,,,,,,,,,,,,,,,,,,
0,0,0,0,T,Y,Green,Triangle,Snake,Finland,Bassoon,...,c389000ab,2f4cb3d51,2,Grandmaster,Cold,h,D,kr,2,2
1,0,1,0,T,Y,Green,Trapezoid,Hamster,Russia,Piano,...,4cd920251,f83c56c21,1,Grandmaster,Hot,a,A,bF,7,8
2,0,0,0,F,Y,Blue,Trapezoid,Lion,Russia,Theremin,...,de9c9f684,ae6800dd0,1,Expert,Lava Hot,h,R,Jc,7,2
3,0,1,0,F,Y,Red,Trapezoid,Snake,Canada,Oboe,...,4ade6ab69,8270f0d71,1,Grandmaster,Boiling Hot,i,D,kW,2,1
4,0,0,0,F,N,Red,Trapezoid,Lion,Canada,Oboe,...,cb43ab175,b164b72a7,1,Grandmaster,Freezing,a,R,qP,7,8


1. bin_3랑 bin_4 피쳐를 T,Y에서 0과 1로

In [5]:
all_data['bin_3'] = all_data['bin_3'].map({'F':0, 'T':1})
all_data['bin_4'] = all_data['bin_4'].map({'N':0, 'Y':1})
#map 유용

In [6]:
all_data['bin_4'].head() #굳

id
0    1
1    1
2    1
3    1
4    0
Name: bin_4, dtype: int64

2. 순서형 피쳐

In [7]:
all_data['ord_0'].head()

id
0    2
1    1
2    1
3    1
4    1
Name: ord_0, dtype: int64

0피처는 이미 숫자로 구성

In [8]:
all_data['ord_1'].head()

id
0    Grandmaster
1    Grandmaster
2         Expert
3    Grandmaster
4    Grandmaster
Name: ord_1, dtype: object

 ord_1과 ord_2에 순서 부여해주면 되겠구나 / map 사용 : T, Y와 비슷

In [9]:
ord1dict = {'Novice':0, 'Contributor':1, 'Expert':2, 'Master':3, 'Grandmaster':4}
ord2dict = {'Freezing':0, 'Cold':1, 'Warm':2, 'Hot':3, 'Boiling Hot':4, 'Lava Hot':5}

all_data['ord_1'] = all_data['ord_1'].map(ord1dict)
all_data['ord_2'] = all_data['ord_2'].map(ord2dict)

In [10]:
all_data['ord_2'].head()

id
0    1
1    3
2    5
3    4
4    0
Name: ord_2, dtype: int64

In [11]:
from sklearn.preprocessing import OrdinalEncoder

ord_345 = ['ord_3', 'ord_4', 'ord_5']

In [12]:
ord_encoder = OrdinalEncoder() #이 객체 집중

all_data[ord_345] = ord_encoder.fit_transform(all_data[ord_345])
#애초에 자동화 가능해서 쓰면 편함 (아스키 순서대로 되는 듯)

for feature, categories in zip(ord_345, ord_encoder.categories_): 
    #_은 사이킷 런 내부에서 인코딩 결과를 의미
    print(feature)
    print(categories)

ord_3
['a' 'b' 'c' 'd' 'e' 'f' 'g' 'h' 'i' 'j' 'k' 'l' 'm' 'n' 'o']
ord_4
['A' 'B' 'C' 'D' 'E' 'F' 'G' 'H' 'I' 'J' 'K' 'L' 'M' 'N' 'O' 'P' 'Q' 'R'
 'S' 'T' 'U' 'V' 'W' 'X' 'Y' 'Z']
ord_5
['AP' 'Ai' 'Aj' 'BA' 'BE' 'Bb' 'Bd' 'Bn' 'CL' 'CM' 'CU' 'CZ' 'Cl' 'DH'
 'DN' 'Dc' 'Dx' 'Ed' 'Eg' 'Er' 'FI' 'Fd' 'Fo' 'GD' 'GJ' 'Gb' 'Gx' 'Hj'
 'IK' 'Id' 'JX' 'Jc' 'Jf' 'Jt' 'KR' 'KZ' 'Kf' 'Kq' 'LE' 'MC' 'MO' 'MV'
 'Mf' 'Ml' 'Mx' 'NV' 'Nf' 'Nk' 'OR' 'Ob' 'Os' 'PA' 'PQ' 'PZ' 'Ps' 'QM'
 'Qb' 'Qh' 'Qo' 'RG' 'RL' 'RP' 'Rm' 'Ry' 'SB' 'Sc' 'TR' 'TZ' 'To' 'UO'
 'Uk' 'Uu' 'Vf' 'Vx' 'WE' 'Wc' 'Wv' 'XI' 'Xh' 'Xi' 'YC' 'Yb' 'Ye' 'ZR'
 'ZS' 'Zc' 'Zq' 'aF' 'aM' 'aO' 'aP' 'ac' 'av' 'bF' 'bJ' 'be' 'cA' 'cG'
 'cW' 'ck' 'cp' 'dB' 'dE' 'dN' 'dO' 'dP' 'dQ' 'dZ' 'dh' 'eG' 'eQ' 'eb'
 'eg' 'ek' 'ex' 'fO' 'fh' 'gJ' 'gM' 'hL' 'hT' 'hh' 'hp' 'iT' 'ih' 'jS'
 'jV' 'je' 'jp' 'kC' 'kE' 'kK' 'kL' 'kU' 'kW' 'ke' 'kr' 'kw' 'lF' 'lL'
 'll' 'lx' 'mb' 'mc' 'mm' 'nX' 'nh' 'oC' 'oG' 'oH' 'oK' 'od' 'on' 'pa'
 'ps' 'qA' 'qJ' 'qK' 'qP' 'qX' '

In [13]:
all_data.head()

,bin_0,bin_1,bin_2,bin_3,bin_4,nom_0,nom_1,nom_2,nom_3,nom_4,...,nom_8,nom_9,ord_0,ord_1,ord_2,ord_3,ord_4,ord_5,day,month
id,,,,,,,,,,,,,,,,,,,,,
0,0,0,0,1,1,Green,Triangle,Snake,Finland,Bassoon,...,c389000ab,2f4cb3d51,2,4,1,7.0,3.0,136.0,2,2
1,0,1,0,1,1,Green,Trapezoid,Hamster,Russia,Piano,...,4cd920251,f83c56c21,1,4,3,0.0,0.0,93.0,7,8
2,0,0,0,0,1,Blue,Trapezoid,Lion,Russia,Theremin,...,de9c9f684,ae6800dd0,1,2,5,7.0,17.0,31.0,7,2
3,0,1,0,0,1,Red,Trapezoid,Snake,Canada,Oboe,...,4ade6ab69,8270f0d71,1,4,4,8.0,3.0,134.0,2,1
4,0,0,0,0,0,Red,Trapezoid,Lion,Canada,Oboe,...,cb43ab175,b164b72a7,1,4,0,0.0,17.0,158.0,7,8


3. 명목형 피처

순서를 무시해도 되기 떄문에 원핫 인코딩 활용


In [14]:
nom_features = ['nom_' + str(i) for i in range(10)]

In [15]:
nom_features

['nom_0',
 'nom_1',
 'nom_2',
 'nom_3',
 'nom_4',
 'nom_5',
 'nom_6',
 'nom_7',
 'nom_8',
 'nom_9']

In [16]:
from sklearn.preprocessing import OneHotEncoder

onehot_encoder = OneHotEncoder()
#all_data[nom_features] = onehot_encoder.fit_transform(all_data[nom_features])

In [17]:
#원핫인코딩 후 열이 늘어날테니까 당연히 안되겠죠 위에처럼은?
#new object needed
encoded_nom_matrix = onehot_encoder.fit_transform(all_data[nom_features])

In [18]:
all_data = all_data.drop(nom_features, axis=1) #drop the ones going to be dropped

4. day and date

In [19]:
date_features = ['day', 'month']

encoded_date_matrix = onehot_encoder.fit_transform(all_data[date_features])

all_data = all_data.drop(date_features, axis=1)

Feature Scailing

In [20]:
from sklearn.preprocessing import MinMaxScaler

ord_features = ['ord_' + str(i) for i in range(6)]

all_data[ord_features] = MinMaxScaler().fit_transform(all_data[ord_features])

COR vs CSR Matrix

In [21]:
#all_data : DataFrame vs encoded_nom_matrix / encoded_date_matrix : CSR Matrix
#transformation needed
from scipy import sparse

all_data_sprs = sparse.hstack([sparse.csr_matrix(all_data), 
                              encoded_nom_matrix,
                              encoded_date_matrix],
                             format='csr')

all_data_sprs

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 9163718 stored elements and shape (500000, 16306)>

In [22]:
num_train = len(train)

X_train = all_data_sprs[:num_train]
X_test = all_data_sprs[num_train:]

y = train['target']

In [23]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(X_train, y, test_size=0.1, 
                                                      stratify=y, random_state=10)

하이퍼파라미터 최적화

In [24]:
%%time

from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression()

#C는 규제 강도를 조절하는 파라미터 
lr_params = {'C':[0.1, 0.125, 0.2], 'max_iter':[800, 900, 1000],
            'solver':['liblinear'], 'random_state':[42]}

gridsearch_logistic_model = GridSearchCV(estimator=logistic_model,
                                        param_grid=lr_params,
                                        scoring='roc_auc',
                                        cv=5)

gridsearch_logistic_model.fit(X_train, y_train)

print('최적하이퍼파라미터: ', gridsearch_logistic_model.best_params_)

최적하이퍼파라미터:  {'C': 0.125, 'max_iter': 800, 'random_state': 42, 'solver': 'liblinear'}
CPU times: user 22min 31s, sys: 3.15 s, total: 22min 34s
Wall time: 5min 50s


In [25]:
y_valid_preds = gridsearch_logistic_model.predict_proba(X_valid)[:, 1]

In [26]:
from sklearn.metrics import roc_auc_score

roc_auc = roc_auc_score(y_valid, y_valid_preds)

print(f'검증 데이터 ROC AUC : {roc_auc:.4f}')

검증 데이터 ROC AUC : 0.8045


In [27]:
y_preds = gridsearch_logistic_model.best_estimator_.predict_proba(X_test)[:,1]

submission['target'] = y_preds
submission.to_csv('submission.csv')